# Binary Spatial Weights Matrices for Si-Gun-Gu and Moran’s I

This notebook loads the NumPy spatial weights arrays for 2010, 2015, 2020, and 2025, and computes Moran’s I for `log(1 + population)` of the 2020 total population by si-gun-gu. `w_mat_` includes marine road bridge and undersea tunnel connections opened by **December 31** of each year, while `w_mat_o_` includes only adjacency from exactly shared boundary vertices (Queen contiguity). The stored arrays are symmetric binary adjacency matrices with entries of 0 or 1. Note that they are not row-standardized.

The 2020 boundaries were finalized by correcting geometries with reference to the 2010 and 2015 boundaries, and the 2025 adjacency was validated against the original boundaries of the same year. For example, in the 2025 data, Guro-gu, Yeongdeungpo-gu, Dongjak-gu, and Gwanak-gu do not meet at a single point; based on earlier cases and the current boundaries, the six neighbor pairs mediated by that shared vertex were retained.


Required additional packages: `pip install libpysal esda`.

References: [libpysal full2W](https://pysal.org/libpysal/stable/generated/libpysal.weights.full2W.html),
[W and row standardization](https://pysal.org/libpysal/stable/generated/libpysal.weights.W.html),
[esda.Moran](https://pysal.org/esda/stable/generated/esda.Moran.html).


In [ ]:
from importlib.resources import files
import json
import numpy as np
import pandas as pd
from libpysal.weights import full2W
from esda import Moran
from pycensuskr import CensusKR

DATA = files("pycensuskr").joinpath("data")

def load_weights(year, queen_only=False):
    name = f"w_mat_{'o_' if queen_only else ''}{year}"
    with DATA.joinpath(name + ".npy").open("rb") as stream:
        matrix = np.load(stream, allow_pickle=False)
    metadata = json.loads(DATA.joinpath(name + "_order.json").read_text())
    codes = metadata["adm2_code_order"]
    return matrix, codes

summary = []
for year in (2010, 2015, 2020, 2025):
    for queen_only in (True, False):
        matrix, codes = load_weights(year, queen_only)
        summary.append({"year": year, "marine_connections": not queen_only,
                        "districts": len(codes), "links": int(matrix.sum() // 2),
                        "islands": int((matrix.sum(axis=1) == 0).sum())})
pd.DataFrame(summary)

,year,marine_connections,districts,links,islands
0,2010,False,251,641,7
1,2010,True,251,655,1
2,2015,False,252,652,8
3,2015,True,252,666,1
4,2020,False,250,640,8
5,2020,True,250,659,1
6,2025,False,252,646,8
7,2025,True,252,664,1


## Aligning observations with the code order

To avoid summing totals with their components, or ADM2 with ADM3, select only **2020 ADM2, all households / total, persons** from the original long table. Rebuild index to the 250 boundary codes (including non-autonomous districts). If any values are missing, exclude those observations together with the corresponding rows and columns of the weights matrix, and print their codes.

In [ ]:
census = CensusKR()
raw = census.load_data(2020)
pop = raw.loc[(raw["type"] == "population") & raw["adm3_code"].isna()
              & (raw["class1"] == "all households") & (raw["class2"] == "total")
              & (raw["unit"] == "persons"), ["adm2_code", "value"]].copy()

pop["adm2_code"] = pd.to_numeric(pop["adm2_code"]).astype("Int64").astype(str)
marine, codes = load_weights(2020)
queen, queen_codes = load_weights(2020, queen_only=True)

boundaries = census.load_districts(2020)
population = pop.set_index("adm2_code")["value"].reindex(codes)
keep = np.isfinite(population.to_numpy(dtype=float))
print("Missing/non-finite population codes excluded:", population.index[~keep].tolist())

y = np.log1p(population[keep].to_numpy(dtype=float))
ids = population.index[keep].tolist()
marine = marine[np.ix_(keep, keep)]
queen = queen[np.ix_(keep, keep)]
print(f"Analysis uses {len(y)} districts; outcome = log(1 + total population).")

Missing/non-finite population codes excluded: []
Analysis uses 250 districts; outcome = log(1 + total population).


## Converting to libpysal W and computing Moran’s I

Pass `ids` to `full2W` in row order. Row standardization is then applied only to the W object. Islands are kept as rows with no neighbors, so their spatial lag is zero. Note that the version with bridge and tunnel connections and the Queen-only version differ in the number of islands and connected components.

Below, Moran's _I_ is computed reproducibly with 999 permutations and a fixed random seed. `p_sim` is the one-sided permutation p-value reported by `esda`.

In [ ]:
results = []
weights = {}
for label, matrix in [("Queen only", queen), ("Queen + marine connections", marine)]:
    # convert the full matrix to a PySAL weights object
    w = full2W(matrix, ids=ids, silence_warnings=True)
    w.transform = "r"
    np.random.seed(2026)
    moran = Moran(y, w, transformation="r", permutations=999)
    weights[label] = w
    results.append({"graph": label, "n": w.n, "components": w.n_components,
                    "island_codes": w.islands, "Moran_I": moran.I,
                    "expected_I": moran.EI, "p_sim_one_tailed": moran.p_sim})
pd.DataFrame(results)

,graph,n,components,island_codes,Moran_I,expected_I,p_sim_one_tailed
0,Queen only,250,10,"[21040, 38090, 38350, 37430, 23310, 23320, 364...",0.524977,-0.004016,0.001
1,Queen + marine connections,250,3,[37430],0.498506,-0.004016,0.001
